# GTEx model building with flashier

💡 **Environment:** `clamp-analyses`  

Empirical Bayes non-negative matrix factorization for single-cell RNA-seq data

## Libraries

In [6]:
if (!requireNamespace("flashier", quietly = TRUE)) {
  install.packages(
  "fastTopics",
  repos = c("https://stephenslab.r-universe.dev", "https://cloud.r-project.org")
  )
  remotes::install_github("willwerscheid/flashier", dependencies = c("Depends","Imports"))
}

In [7]:
library(here)
library(flashier)
library(ebnm)
set.seed(3)

## Input

In [ ]:
gtex_data <- readRDS(here("output/gtex/df_gtex_fbm_filt.rds"))
K <- readRDS(here("output/gtex/CLAMP_K_gtex.rds"))
K <- as.integer(K)

In [9]:
# flashier wants samples x genes
X <- t(as.matrix(gtex_data))

stopifnot(is.numeric(K), length(K) == 1)
stopifnot(K <= min(nrow(X), ncol(X)))

# Run Flashier

In [11]:
fl <- flash_init(X, var_type = 0L)

fl <- flash_greedy(
  fl,
  Kmax = as.integer(K),
  ebnm_fn = c(ebnm_point_exponential, ebnm_point_normal),
  verbose = 1L
)

Adding factor 1 to flash object...
Adding factor 2 to flash object...
Adding factor 3 to flash object...
Adding factor 4 to flash object...
Adding factor 5 to flash object...
Adding factor 6 to flash object...
Adding factor 7 to flash object...
Adding factor 8 to flash object...
Adding factor 9 to flash object...
Adding factor 10 to flash object...
Adding factor 11 to flash object...
Adding factor 12 to flash object...
Adding factor 13 to flash object...
Adding factor 14 to flash object...
Adding factor 15 to flash object...
Adding factor 16 to flash object...
Adding factor 17 to flash object...
Adding factor 18 to flash object...
Adding factor 19 to flash object...
Adding factor 20 to flash object...
Adding factor 21 to flash object...
Adding factor 22 to flash object...
Adding factor 23 to flash object...
Adding factor 24 to flash object...
Adding factor 25 to flash object...
Adding factor 26 to flash object...
Adding factor 27 to flash object...
Adding factor 28 to flash object...
A

In [12]:
fl <- flash_backfit(fl, verbose = 1L, maxiter = 20)

Backfitting 412 factors (tolerance: 5.60e+00)...
  Difference between iterations is within 1.0e+07...
  Difference between iterations is within 1.0e+06...
  Difference between iterations is within 1.0e+05...
  Difference between iterations is within 1.0e+04...
  --Maximum number of iterations reached!
Wrapping up...
Done.


In [19]:
fit <- flash_fit(fl)

pm1 <- flash_fit_get_pm(fit, n = 1)
F_scores <- pm1

B <- t(F_scores)  # K x samples

sample_names <- colnames(gtex_data)

stopifnot(nrow(F_scores) == length(sample_names))

colnames(F_scores) <- paste0("LV", seq_len(ncol(F_scores)))
rownames(F_scores) <- sample_names

rownames(B) <- colnames(F_scores)
colnames(B) <- sample_names


In [21]:
head(B)
dim(B)

,GTEX-1117F-0226-SM-5GZZ7,GTEX-1117F-0426-SM-5EGHI,GTEX-1117F-0526-SM-5EGHJ,GTEX-1117F-0626-SM-5N9CS,GTEX-1117F-0726-SM-5GIEN,GTEX-1117F-1326-SM-5EGHH,GTEX-1117F-2426-SM-5EGGH,GTEX-1117F-2526-SM-5GZY6,GTEX-1117F-2826-SM-5GZXL,GTEX-1117F-2926-SM-5GZYI,⋯,GTEX-ZZPU-1126-SM-5N9CW,GTEX-ZZPU-1226-SM-5N9CK,GTEX-ZZPU-1326-SM-5GZWS,GTEX-ZZPU-1426-SM-5GZZ6,GTEX-ZZPU-1826-SM-5E43L,GTEX-ZZPU-2126-SM-5EGIU,GTEX-ZZPU-2226-SM-5EGIV,GTEX-ZZPU-2426-SM-5E44I,GTEX-ZZPU-2626-SM-5E45Y,GTEX-ZZPU-2726-SM-5NQ8O
LV1,1.005020e+00,9.989777e-06,7.404323e-01,1.176645e+00,2.403937e-05,1.743197e-05,1.596612e+00,8.246681e-01,4.603512e-01,2.839578e-01,⋯,1.889122e-05,2.496831e-01,1.047128e+00,1.719564e-05,1.326522e-01,1.150673e+00,1.601012e-05,3.271694e-01,2.136395e-05,2.844770e-01
LV2,2.693017e-05,7.442377e-01,1.758844e-05,9.683399e-06,7.534418e-01,1.388982e-01,2.247298e-05,6.930963e-06,1.259937e-05,1.217350e-05,⋯,5.240411e-01,7.721273e-06,9.254178e-02,2.537854e-01,1.683423e-05,9.037751e-06,2.164500e-01,1.051145e-05,8.799499e-01,1.216220e-05
LV3,3.770630e-01,4.860916e-05,1.460846e-01,3.211615e-02,7.275764e-02,4.306669e-01,2.386915e-02,9.074504e-01,2.860994e-01,7.844653e-01,⋯,2.553809e-04,6.358149e-02,5.968605e-02,1.346731e-01,1.297387e-04,7.870707e-05,8.702965e-01,1.061771e-03,8.353020e-05,1.847161e-01
LV4,3.085676e-02,1.659679e-04,2.450358e-05,3.356754e-05,1.336897e-05,4.449379e-05,1.889484e-05,1.617181e-01,1.692742e-05,1.182539e-05,⋯,6.079626e-06,6.956046e-05,1.213974e-05,1.204938e-05,9.135659e-06,8.822347e-06,6.870444e-05,3.253867e-06,1.768252e-05,5.706735e-05
LV5,2.874569e-01,1.125830e+00,5.946417e-01,3.244018e-01,4.176809e-01,2.391158e-01,1.439017e-01,1.321937e-01,2.401397e-01,2.095867e-01,⋯,7.346510e-01,2.416913e-01,6.675461e-05,1.182693e-01,8.194852e-01,1.484852e-01,2.649336e-01,9.947373e-01,1.532512e+00,6.500608e-01
LV6,1.782100e-05,5.395976e-01,2.080636e-05,3.295017e-05,4.343588e-05,4.915532e-05,3.635711e-05,9.231465e-01,1.716422e-05,7.178377e-01,⋯,1.724921e-05,5.889014e-05,6.837117e-02,1.715078e-01,2.852306e-05,4.895754e-04,1.026962e+00,1.762771e-04,5.194164e-01,2.669121e-05


[1]   412 17382

In [22]:
output_dir <- here("output/gtex/flashier")
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

write.csv(
  B,
  file = file.path(output_dir, "gtex_B.csv"),
  quote = FALSE
)

In [23]:
saveRDS(fl, file.path(output_dir, "flashier_model.rds"))